In [ ]:
# ── CELL 1: INITIALIZE SPARK SESSION ──────────────────────────────────────
import os
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession locally to avoid Python version mismatch with external cluster
spark = SparkSession.builder \
    .appName("olist-notebook-analysis") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise inside the notebook
spark.sparkContext.setLogLevel("WARN")

print("Spark Session created successfully!")
print("Spark Master URI:", spark.sparkContext.master)

🎯 Spark Session created successfully!
Spark Master URI: local[*]


In [3]:
# ── CELL 2: READ AND DISPLAY DATA ─────────────────────────────────────────
# 1. Read the dataset in Delta format from Bronze layer in MinIO
df_orders = (
    spark.read \
    .format("delta") \
    .load("s3a://bronze/csv/orders/")
)

# 2. Display the first 10 rows inside the notebook
display(df_orders.limit(10))

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, _ingested_at: timestamp, _source_file: string]

## 2. Data Profiling: Orders Dataset
In this section, we perform an initial audit specifically for the **Orders** dataset. The goal is to evaluate the data quality by inspecting its schema, calculating summary statistics, and identifying missing values or duplicates before proceeding to the transformation phase.

In [4]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Schema Enforcement
Apply strict type casting for order-related columns to ensure data consistency.

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType

# 1. Define target data types
target_types = {
    "order_id": StringType(),
    "customer_id": StringType(),
    "order_status": StringType(),
    "order_purchase_timestamp": TimestampType(),
    "order_approved_at": TimestampType(),
    "order_delivered_carrier_date": TimestampType(),
    "order_delivered_customer_date": TimestampType(),
    "order_estimated_delivery_date": TimestampType()
}

# 2. Iterate and cast columns
for col_name, data_type in target_types.items():
    if col_name in df_orders.columns:
        df_orders = df_orders.withColumn(col_name, F.col(col_name).cast(data_type))

# 3. Validate final schema
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [6]:
display(df_orders.describe())

DataFrame[summary: string, order_id: string, customer_id: string, order_status: string, _source_file: string]

In [7]:
# Checking for redundant records within the Orders dataset
total_count = df_orders.count()
distinct_count = df_orders.distinct().count()

duplicate_count = total_count - distinct_count
print(f"Total Duplicates: {duplicate_count}")

Total Duplicates: 0


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Orphaned Records Imputation
Identify missing or orphaned customer IDs and impute them to maintain referential integrity.

In [8]:
from pyspark.sql import functions as F

# 1. Identify orphaned IDs via left anti join
# 1. Read from the Bronze Layer since Silver is not saved yet
df_customers = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/customers/")
)
orphaned_customers = df_orders.select("customer_id").distinct() \
    .join(df_customers.select("customer_id"), "customer_id", "left_anti")

# 2. Collect orphans for imputation
orphaned_customer_ids = [row["customer_id"] for row in orphaned_customers.collect()]
print(f"Data Quality Check: Found {len(orphaned_customer_ids)} orphaned Customer IDs.")

# 3. Impute null/orphaned IDs with "-1"
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(
        F.col("customer_id").isNull() | F.col("customer_id").isin(orphaned_customer_ids),
        F.lit("-1")
    ).otherwise(F.col("customer_id"))
)

# 4. Audit orphaned records
if len(orphaned_customer_ids) > 0:
    order_errors_audit = df_orders.filter(F.col("customer_id") == "-1") \
        .select("order_id", "customer_id") \
        .withColumn("error_reason", F.lit("Orphaned or Null customer_id - Imputed with -1"))
    
    print("🚩 Success: Errors logged into the Audit system.")

# 5. Validate schema
df_orders.printSchema()

Data Quality Check: Found 0 orphaned Customer IDs.
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [9]:
# Analyzing data completeness by counting NULL values in each column of the Orders tablefrom pyspark.sql import functions as F
from pyspark.sql import functions as F
null_counts = df_orders.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_orders.columns])

display(null_counts)

DataFrame[order_id: bigint, customer_id: bigint, order_status: bigint, order_purchase_timestamp: bigint, order_approved_at: bigint, order_delivered_carrier_date: bigint, order_delivered_customer_date: bigint, order_estimated_delivery_date: bigint, _ingested_at: bigint, _source_file: bigint]

## 3. Advanced Exploratory Data Analysis (EDA): Null Pattern Investigation
In this section, we move beyond simple null counting to analyze the patterns behind missing values. By correlating missing timestamps with the `order_status`, we can determine if the NULLs are expected (e.g., a "shipped" order shouldn't have a delivery date) or represent data integrity issues.

In [10]:
from pyspark.sql import functions as F

def analyze_null_status_distribution(df, date_column):
    """
    Analyzes and displays the distribution of order statuses for records where 
    a specific date column is NULL, providing insights into data consistency.
    """
    print(f"\n>>> Analysis for Null values in: {date_column} <<<")
    
    # 1. Filter records where the target date column is NULL
    df_null = df.filter(F.col(date_column).isNull())
    
    # 2. Aggregate counts by order status to identify patterns
    status_summary = df_null.groupby("order_status").count().orderBy(F.desc("count"))
    
    print(f"Distribution of order statuses for {date_column}:")
    status_summary.show()
    
    # 3. Display a sample of the data for manual inspection
    print(f"Sample data for Null {date_column}:")
    display(df_null.select("order_id", "order_status", "order_purchase_timestamp", date_column).limit(5))
    print("="*50)

# Iterate through key date columns to execute the distribution analysis
date_cols = ["order_delivered_customer_date", "order_delivered_carrier_date", "order_approved_at"]

for col in date_cols:
    analyze_null_status_distribution(df_orders, col)


>>> Analysis for Null values in: order_delivered_customer_date <<<
Distribution of order statuses for order_delivered_customer_date:
+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  619|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|   delivered|    8|
|     created|    5|
|    approved|    2|
+------------+-----+

Sample data for Null order_delivered_customer_date:


DataFrame[order_id: string, order_status: string, order_purchase_timestamp: timestamp, order_delivered_customer_date: timestamp]


>>> Analysis for Null values in: order_delivered_carrier_date <<<
Distribution of order statuses for order_delivered_carrier_date:
+------------+-----+
|order_status|count|
+------------+-----+
| unavailable|  609|
|    canceled|  550|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
|   delivered|    2|
+------------+-----+

Sample data for Null order_delivered_carrier_date:


DataFrame[order_id: string, order_status: string, order_purchase_timestamp: timestamp, order_delivered_carrier_date: timestamp]


>>> Analysis for Null values in: order_approved_at <<<
Distribution of order statuses for order_approved_at:
+------------+-----+
|order_status|count|
+------------+-----+
|    canceled|  141|
|   delivered|   14|
|     created|    5|
+------------+-----+

Sample data for Null order_approved_at:


DataFrame[order_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp]

### 3.1. Data Integrity Validation: Delivered Orders vs. Missing Timestamps
This function performs a targeted audit on "delivered" orders to identify logical inconsistencies where the status is marked as delivered but essential timestamps are missing. Calculating the error ratio helps determine the severity of data loss in the source system.

In [11]:
def check_delivered_error_ratio(df, column_name):
    """
    Calculates the count and percentage of 'delivered' status records 
    that contain NULL values in a specified date column.
    """
    total_count = df.count()
    
    # Filter records that violate logical consistency (Delivered status but missing date)
    error_df = df.filter((F.col("order_status") == "delivered") & (F.col(column_name).isNull()))
    error_count = error_df.count()
    
    error_ratio = (error_count / total_count) * 100
    
    print(f"Column: {column_name}")
    print(f"- Error Count: {error_count}")
    print(f"- Error Ratio: {error_ratio:.4f}%")
    print("-" * 30)
    
    return error_count, error_ratio

# Execution: Analyze integrity across key delivery-related columns
for col in ["order_delivered_customer_date", "order_delivered_carrier_date", "order_approved_at"]:
    check_delivered_error_ratio(df_orders, col)

Column: order_delivered_customer_date
- Error Count: 8
- Error Ratio: 0.0080%
------------------------------
Column: order_delivered_carrier_date
- Error Count: 2
- Error Ratio: 0.0020%
------------------------------
Column: order_approved_at
- Error Count: 14
- Error Ratio: 0.0141%
------------------------------


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Sequential Timestamp Imputation
Calculate data-driven median deltas and apply a sequential imputation pipeline to recover missing order timestamps, ensuring full auditability.

In [12]:
from pyspark.sql import functions as F

# ==========================================
# STEP 1: CALCULATE GLOBAL MEDIAN DELTAS (DATA-DRIVEN)
# ==========================================
# Filtering completely valid rows to calculate true historical business deltas
valid_timestamps_df = df_orders.filter(
    F.col("order_purchase_timestamp").isNotNull() & 
    F.col("order_approved_at").isNotNull() & 
    F.col("order_delivered_carrier_date").isNotNull() & 
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_estimated_delivery_date").isNotNull()
).select(
    (F.unix_timestamp("order_approved_at") - F.unix_timestamp("order_purchase_timestamp")).alias("p_to_a"),
    (F.unix_timestamp("order_delivered_carrier_date") - F.unix_timestamp("order_approved_at")).alias("a_to_c"),
    (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_delivered_carrier_date")).alias("c_to_cust"),
    (F.unix_timestamp("order_estimated_delivery_date") - F.unix_timestamp("order_purchase_timestamp")).alias("p_to_e")
)

# Calculate medians (0.5 quantile)
medians = valid_timestamps_df.approxQuantile(["p_to_a", "a_to_c", "c_to_cust", "p_to_e"], [0.5], 0.01)

med_p_to_a   = int(medians[0][0]) if (medians and medians[0]) else 3600
med_a_to_c   = int(medians[1][0]) if (medians and medians[1]) else 86400
med_c_to_cust = int(medians[2][0]) if (medians and medians[2]) else 259200
med_p_to_e   = int(medians[3][0]) if (medians and medians[3]) else 950400

print("Data-Driven Medians calculated successfully.")


# ==========================================
# STEP 2: CAPTURE FULL ROW AUDIT TRAIL BEFORE IMPUTATION
# ==========================================
is_imputable_order = (F.col("order_status") == "delivered") & (
    F.col("order_purchase_timestamp").isNull() |
    F.col("order_approved_at").isNull() | 
    F.col("order_delivered_carrier_date").isNull() | 
    F.col("order_delivered_customer_date").isNull()
)

# Capturing the FULL row with all original columns + error reason from df_orders
order_errors_audit = df_orders.filter(is_imputable_order) \
    .withColumn("error_reason", F.lit("Delivered status with missing timestamps - Fixed via Sequential Imputation"))


# ==========================================
# STEP 3: SEQUENTIAL BACKWARD/FORWARD IMPUTATION PIPELINE
# ==========================================
# The output is now saved directly into orders_cleaned_df

# 3.1 Impute Missing 'order_purchase_timestamp' (Using fallback chain)
orders_cleaned_df = df_orders.withColumn(
    "order_purchase_timestamp",
    F.when(F.col("order_purchase_timestamp").isNotNull(), F.col("order_purchase_timestamp"))
     .when(F.col("order_approved_at").isNotNull(), F.from_unixtime(F.unix_timestamp("order_approved_at") - med_p_to_a).cast("timestamp"))
     .when(F.col("order_delivered_carrier_date").isNotNull(), F.from_unixtime(F.unix_timestamp("order_delivered_carrier_date") - (med_p_to_a + med_a_to_c)).cast("timestamp"))
     .when(F.col("order_delivered_customer_date").isNotNull(), F.from_unixtime(F.unix_timestamp("order_delivered_customer_date") - (med_p_to_a + med_a_to_c + med_c_to_cust)).cast("timestamp"))
     .otherwise(F.from_unixtime(F.unix_timestamp("order_estimated_delivery_date") - med_p_to_e).cast("timestamp"))
)

# 3.2 Impute Missing 'order_approved_at'
orders_cleaned_df = orders_cleaned_df.withColumn(
    "order_approved_at",
    F.when(F.col("order_approved_at").isNotNull(), F.col("order_approved_at"))
     .when(F.col("order_status") == "delivered", F.from_unixtime(F.unix_timestamp("order_purchase_timestamp") + med_p_to_a).cast("timestamp"))
     .otherwise(F.col("order_approved_at"))
)

# 3.3 Impute Missing 'order_delivered_carrier_date'
orders_cleaned_df = orders_cleaned_df.withColumn(
    "order_delivered_carrier_date",
    F.when(F.col("order_delivered_carrier_date").isNotNull(), F.col("order_delivered_carrier_date"))
     .when(F.col("order_status") == "delivered", F.from_unixtime(F.unix_timestamp("order_approved_at") + med_a_to_c).cast("timestamp"))
     .otherwise(F.col("order_delivered_carrier_date"))
)

# 3.4 Impute Missing 'order_delivered_customer_date'
orders_cleaned_df = orders_cleaned_df.withColumn(
    "order_delivered_customer_date",
    F.when(F.col("order_delivered_customer_date").isNotNull(), F.col("order_delivered_customer_date"))
     .when(F.col("order_status") == "delivered", F.from_unixtime(F.unix_timestamp("order_delivered_carrier_date") + med_c_to_cust).cast("timestamp"))
     .otherwise(F.col("order_delivered_customer_date"))
)

print(f"Imputation completed. Total rows logged to audit: {order_errors_audit.count()}")
print(f"Cleaned DataFrame created successfully as 'orders_cleaned_df' with {orders_cleaned_df.count()} rows.")

Data-Driven Medians calculated successfully.
Imputation completed. Total rows logged to audit: 23
Cleaned DataFrame created successfully as 'orders_cleaned_df' with 99441 rows.


### 4.3. Post-Cleaning Validation: Missing Values Audit
After isolating inconsistent records, we perform a follow-up audit on the `orders_cleaned_df` to verify the state of the remaining missing values. This step ensures that our cleaning logic has effectively addressed the targeted data quality issues before we proceed to further transformations.

In [13]:
# 1. Recalculating NULL values across all columns for the cleaned dataset
from pyspark.sql import functions as F

# Aggregating null counts to verify data completeness after the quarantine process
null_counts = orders_cleaned_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in orders_cleaned_df.columns])

# 2. Display the final null distribution for the Orders Silver Layer
display(null_counts)

DataFrame[order_id: bigint, customer_id: bigint, order_status: bigint, order_purchase_timestamp: bigint, order_approved_at: bigint, order_delivered_carrier_date: bigint, order_delivered_customer_date: bigint, order_estimated_delivery_date: bigint, _ingested_at: bigint, _source_file: bigint]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### 4.3. Logical Consistency Check: Date Chronology Validation
In this step, we verify the temporal logic of the order lifecycle. We identify records where timestamps appear out of order—such as a shipping date occurring before the purchase date—to ensure that the sequence of events (Purchase → Approval → Carrier → Customer) is chronologically sound.
Then, enforce the logical order of order lifecycle events (Purchase → Approved → Carrier → Customer) and log violations to the audit system.

In [14]:
from pyspark.sql import functions as F

# ==========================================
# STEP 1: DEFINE LOGICAL VIOLATIONS IN THE DATE SEQUENCE
# ==========================================
is_chronology_error = (
    # Approval cannot happen before Purchase
    (F.col("order_approved_at") < F.col("order_purchase_timestamp")) |
    
    # Carrier (Shipment) cannot happen before Approval or Purchase
    (F.col("order_delivered_carrier_date") < F.col("order_approved_at")) |
    (F.col("order_delivered_carrier_date") < F.col("order_purchase_timestamp")) |
    
    # Delivery to Customer cannot happen before Shipment, Approval, or Purchase
    (F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date")) |
    (F.col("order_delivered_customer_date") < F.col("order_approved_at")) |
    (F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"))
)

# ==========================================
# STEP 2: CAPTURE FULL ROW CHRONOLOGY ERRORS & UNION TO EXISTING AUDIT
# ==========================================
# 2.1 Filter full rows that violate chronology and add the specific reason
new_chronology_errors = orders_cleaned_df.filter(is_chronology_error) \
    .withColumn("error_reason", F.lit("Invalid date chronology (Olist logic violation)"))

chronology_error_count = new_chronology_errors.count()
print(f"Data Quality Metrics: Found {chronology_error_count} records with chronological errors.")

# 2.2 Dynamic Union: Append these new errors into our master order_errors_audit DataFrame
if chronology_error_count > 0:
    # We use unionByName to ensure columns align perfectly regardless of their order
    order_errors_audit = order_errors_audit.unionByName(new_chronology_errors, allowMissingColumns=True)
    print("Success: Chronology errors successfully appended to the Master Order Audit DataFrame.")


# ==========================================
# STEP 3: CHRONOLOGICAL ORDER ENFORCEMENT (FIXING THE DATES)
# ==========================================
# Directly correcting the timestamps in-place within orders_cleaned_df
# Rule: Ensure Timeline logic holds true (Purchase <= Approved <= Carrier <= Customer)

# 3.1 Fix Approved At: If it's before purchase, set it equal to purchase timestamp
orders_cleaned_df = orders_cleaned_df.withColumn(
    "order_approved_at",
    F.when(F.col("order_approved_at") < F.col("order_purchase_timestamp"), F.col("order_purchase_timestamp"))
     .otherwise(F.col("order_approved_at"))
)

# 3.2 Fix Carrier Date: If it's before approval, push it to match approval timestamp
orders_cleaned_df = orders_cleaned_df.withColumn(
    "order_delivered_carrier_date",
    F.when(F.col("order_delivered_carrier_date") < F.col("order_approved_at"), F.col("order_approved_at"))
     .otherwise(F.col("order_delivered_carrier_date"))
)

# 3.3 Fix Delivered Customer Date: If it's before carrier, push it to match carrier timestamp
orders_cleaned_df = orders_cleaned_df.withColumn(
    "order_delivered_customer_date",
    F.when(F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"), F.col("order_delivered_carrier_date"))
     .otherwise(F.col("order_delivered_customer_date"))
)

print("Success: Chronological dates adjusted successfully in orders_cleaned_df.")


# ==========================================
# STEP 4: CLEANUP & MEMORY OPTIMIZATION (DROP OLD AUDIT TABLE VARIABLES)
# ==========================================
# Completely dropping the temporary table variable from the old logic to free up memory
if 'orders_chronology_errors_df' in locals() or 'orders_chronology_errors_df' in globals():
    del orders_chronology_errors_df

print(f"\nFinal Master Audit Log Count (Imputation + Chronology Errors): {order_errors_audit.count()} rows.")

Data Quality Metrics: Found 1382 records with chronological errors.
Success: Chronology errors successfully appended to the Master Order Audit DataFrame.
Success: Chronological dates adjusted successfully in orders_cleaned_df.

Final Master Audit Log Count (Imputation + Chronology Errors): 1405 rows.


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Feature Engineering
Calculate key performance metrics (lead time, shipping buffers) and classify delivery status to evaluate operational efficiency.

In [15]:
from pyspark.sql import functions as F

# ==========================================
# 1. Calculate durations and business buffers
# ==========================================
orders_detailed = orders_cleaned_df.withColumn(
    "handling_days", 
    F.datediff("order_delivered_carrier_date", "order_approved_at")
).withColumn(
    "shipping_days", 
    F.datediff("order_delivered_customer_date", "order_delivered_carrier_date")
).withColumn(
    "total_lead_time", 
    F.datediff("order_delivered_customer_date", "order_purchase_timestamp")
).withColumn(
    "days_diff_estimated", 
    F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")
).withColumn(
    "estimated_buffer", # Measuring the safety margin between purchase and promise
    F.datediff("order_estimated_delivery_date", "order_purchase_timestamp")
)

# ==========================================
# 2. Performance classification and magnitude analysis
# ==========================================
orders_detailed = orders_detailed.withColumn(
    "delivery_status_detail",
    F.when(F.col("order_status") == "canceled", "Canceled")
     .when(F.col("order_status") == "unavailable", "Unavailable")
     .when(F.col("order_status").isin("shipped", "processing", "approved", "created","invoiced"), "In Progress")
     .when(F.col("order_status") == "delivered", 
           F.when(F.col("days_diff_estimated") < 0, "Early")
            .when(F.col("days_diff_estimated") == 0, "On Time")
            .when(F.col("days_diff_estimated") > 0, "Late")
            .otherwise(None)
     )
     .otherwise("Other")
).withColumn(
    "abs_days_diff", 
    # [FIXED] Enclosed conditions within parentheses () to correct operator precedence
    F.when((F.col("order_status") == "delivered") & (F.col("days_diff_estimated").isNotNull()), 
           F.abs(F.col("days_diff_estimated")))
     .otherwise(F.lit(None))
)

# ==========================================
# 3. Preview final metrics for business validation
# ==========================================
print("=== Delivery Status Detail Breakdown ===")
orders_detailed.groupBy("order_status", "delivery_status_detail").count().show()

print("=== Previewing Sample Data ===")
orders_detailed.select(
    "order_id", 
    "order_status",
    "total_lead_time", 
    "days_diff_estimated", 
    "estimated_buffer",
    "delivery_status_detail",
    "abs_days_diff"
).show(10)

=== Delivery Status Detail Breakdown ===
+------------+----------------------+-----+
|order_status|delivery_status_detail|count|
+------------+----------------------+-----+
|   delivered|                 Early|88649|
|   delivered|               On Time| 1293|
|     shipped|           In Progress| 1107|
|    invoiced|           In Progress|  314|
|   delivered|                  Late| 6536|
|     created|           In Progress|    5|
|    approved|           In Progress|    2|
|    canceled|              Canceled|  625|
|  processing|           In Progress|  301|
| unavailable|           Unavailable|  609|
+------------+----------------------+-----+

=== Previewing Sample Data ===
+--------------------+------------+---------------+-------------------+----------------+----------------------+-------------+
|            order_id|order_status|total_lead_time|days_diff_estimated|estimated_buffer|delivery_status_detail|abs_days_diff|
+--------------------+------------+---------------+--------

In [16]:
orders_detailed.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- handling_days: integer (nullable = true)
 |-- shipping_days: integer (nullable = true)
 |-- total_lead_time: integer (nullable = true)
 |-- days_diff_estimated: integer (nullable = true)
 |-- estimated_buffer: integer (nullable = true)
 |-- delivery_status_detail: string (nullable = true)
 |-- abs_days_diff: integer (nullable = true)



### 5.2. Quick Insight: Late Delivery Magnitude
Before moving to the next stage, we calculate the average and median delay for orders flagged as "Late". This provides an immediate business pulse on the severity of logistics bottlenecks.

In [17]:
from pyspark.sql import functions as F

# 1. Filter only for truly "Late" orders, since chronological anomalies are already fixed 🛡️
late_orders_summary = orders_detailed.filter(
        F.col("delivery_status_detail") == "Late"
    ) \
    .select(
        F.round(F.avg("abs_days_diff"), 0).alias("average_delay_days"),
        F.percentile_approx("abs_days_diff", 0.5).alias("median_delay_days")
    )

# 2. Output the summary metrics for business validation
late_orders_summary.show()

+------------------+-----------------+
|average_delay_days|median_delay_days|
+------------------+-----------------+
|              11.0|                7|
+------------------+-----------------+



### Final Schema Enforcement & Integrity Injection
Cast data types to match the target schema, select required columns, and inject a defensive '-1' record to maintain referential integrity.

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, IntegerType
import decimal

# ==========================================
# STEP 1: STRICT TYPE CASTING & SCHEMA ENFORCEMENT
# ==========================================
# We cast every single column explicitly to match your target data types perfectly

orders_detailed = orders_detailed \
    .withColumn("order_id", F.col("order_id").cast(StringType())) \
    .withColumn("customer_id", F.col("customer_id").cast(StringType())) \
    .withColumn("order_status", F.col("order_status").cast(StringType())) \
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast(TimestampType())) \
    .withColumn("order_approved_at", F.col("order_approved_at").cast(TimestampType())) \
    .withColumn("order_delivered_carrier_date", F.col("order_delivered_carrier_date").cast(TimestampType())) \
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast(TimestampType())) \
    .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast(TimestampType())) \
    .withColumn("handling_days", F.col("handling_days").cast(IntegerType())) \
    .withColumn("shipping_days", F.col("shipping_days").cast(IntegerType())) \
    .withColumn("total_lead_time", F.col("total_lead_time").cast(IntegerType())) \
    .withColumn("days_diff_estimated", F.col("days_diff_estimated").cast(IntegerType())) \
    .withColumn("estimated_buffer", F.col("estimated_buffer").cast(IntegerType())) \
    .withColumn("delivery_status_detail", F.col("delivery_status_detail").cast(StringType())) \
    .withColumn("abs_days_diff", F.col("abs_days_diff").cast(IntegerType()))

# ==========================================
# STEP 2: SELECT & ALIGN ONLY TARGET COLUMNS
# ==========================================
# This ensures no extra columns from previous experiments remain in the final DataFrame
target_columns = [
    "order_id", "customer_id", "order_status", 
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", 
    "order_delivered_customer_date", "order_estimated_delivery_date",
    "handling_days", "shipping_days", "total_lead_time", "days_diff_estimated", "estimated_buffer",
    "delivery_status_detail", "abs_days_diff"
]

orders_final = orders_detailed.select(*target_columns)

# ==========================================
# STEP 3: CREATE AND INJECT THE '-1' RECORD BASED ON THE PERFECT SCHEMA
# ==========================================
# 3.1 Prepare safe default values for the -1 record matching the data types
unknown_order_values = [(
    "-1",             # order_id (string)
    "-1",             # customer_id (string)
    "unknown",        # order_status (string)
    None,             # order_purchase_timestamp (timestamp)
    None,             # order_approved_at (timestamp)
    None,             # order_delivered_carrier_date (timestamp)
    None,             # order_delivered_customer_date (timestamp)
    None,             # order_estimated_delivery_date (timestamp)
    0,                # handling_days (integer)
    0,                # shipping_days (integer)
    0,                # total_lead_time (integer)
    0,                # days_diff_estimated (integer)
    0,                # estimated_buffer (integer)
    "Unknown Order",  # delivery_status_detail (string)
    0                 # abs_days_diff (integer)
)]

# 3.2 Create the temporary DataFrame for the -1 row using the now-perfect schema
df_unknown_row = spark.createDataFrame(unknown_order_values, schema=orders_final.schema)

# 3.3 Protection: Remove the -1 row if it was already appended in a previous run to avoid duplicates
orders_final = orders_final.filter(F.col("order_id") != "-1")

# 3.4 Append the -1 row safely
orders_final = orders_final.unionByName(df_unknown_row)

print("🎯 Schema Enforced & Cleaned! The '-1' Record has been safely injected.")

# ==========================================
# STEP 4: FINAL SCHEMA VERIFICATION PRINTING
# ==========================================
print("\n=== FINAL SILVER ORDERS SCHEMA VALIDATION ===")
orders_final.printSchema()

# Show the integrity row to be absolutely sure
print("=== Validation: Verification of the Imputed Integrity Row ===")
orders_final.filter(F.col("order_id") == "-1").show()

🎯 Schema Enforced & Cleaned! The '-1' Record has been safely injected.

=== FINAL SILVER ORDERS SCHEMA VALIDATION ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- handling_days: integer (nullable = true)
 |-- shipping_days: integer (nullable = true)
 |-- total_lead_time: integer (nullable = true)
 |-- days_diff_estimated: integer (nullable = true)
 |-- estimated_buffer: integer (nullable = true)
 |-- delivery_status_detail: string (nullable = true)
 |-- abs_days_diff: integer (nullable = true)

=== Validation: Verification of the Imputed Integrity Row ===
+--------+-----------+------------+---------

In [19]:
# ── Define target storage paths on MinIO ──────────────────────────────────
SILVER_ORDERS_PATH = "s3a://silver/refined/orders/"
AUDIT_LOG_PATH     = "s3a://silver/qa_issues/order_errors_audit/"

# ── 1. Write the final refined orders DataFrame to Silver layer ───────────
orders_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_ORDERS_PATH)

print(f"Successfully saved orders_final to: {SILVER_ORDERS_PATH}")

# ── 2. Write the QA error logs DataFrame to the Audit path ────────────────
order_errors_audit.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(AUDIT_LOG_PATH)

# Explicitly refresh the Delta cache for this path to make updates visible immediately
spark.catalog.refreshByPath(AUDIT_LOG_PATH)

print(f"Successfully saved order_errors_audit to: {AUDIT_LOG_PATH}")

Successfully saved orders_final to: s3a://silver/refined/orders/
Successfully saved order_errors_audit to: s3a://silver/qa_issues/order_errors_audit/


In [20]:
###########################################################################################################################################################